[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/010049nn/EZ_pair_graph/blob/main/demo_ez_pair_graph.ipynb)

# EZ-Pair Graph Demo Notebook

**Scalable unified-axis visualization for summarizing large-scale paired data**

This notebook demonstrates how to use the `ez-pair-graph` Python package for visualizing paired data. EZ-Pair Graph provides four complementary visualization types:

1. **Slope Graph** — Individual paired observations as connecting lines
2. **Trapezoid Plot** — Quartile-based summary bands for ascending/descending groups  
3. **Clustered Line Plot** — Cluster-level summaries with half-boxplots
4. **Parallel Arrow Plot** — Direction and magnitude of cluster-level changes

**Reference:** Ezoe, A., Seki, M., & Mochida, K. (2026). EZ-Pair Graph: Scalable unified-axis visualization for summarizing large-scale paired data. *Bioinformatics Advances*.

**License:** Academic and non-commercial research use only (RIKEN). The technology is currently under patent application.

> **Google Colab で実行する場合:** 下のセルを最初に実行してください。
> ローカル環境で既に  済みの場合はスキップしてください。

In [ ]:
# === Google Colab セットアップ ===
# GitHubリポジトリから直接インストール
!pip install git+https://github.com/010049nn/EZ_pair_graph.git -q

# テストデータをダウンロード
import urllib.request, os
base = "https://raw.githubusercontent.com/010049nn/EZ_pair_graph/main/"
for f in ["test_dataset.txt", "test_dataset_n1000.txt"]:
    if not os.path.exists(f):
        urllib.request.urlretrieve(base + f, f)
        print(f"Downloaded {f}")
print("Setup complete!")

## 1. Installation

```bash
pip install ez-pair-graph
```

Or install from source:
```bash
git clone https://github.com/010049nn/EZ_pair_graph.git
cd EZ_pair_graph
pip install -e .
```

In [ ]:
import ez_pair_graph as ezpg
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

print(f"ez-pair-graph version: {ezpg.__version__}")

## 2. Quick Start with Synthetic Data

### 2.1 Generate paired data with a known shift

We create synthetic paired observations where group B values are shifted upward relative to group A, simulating a treatment effect.

In [ ]:
# Generate synthetic paired data
np.random.seed(42)
n = 300

# Group A: normal distribution (mean=50, SD=10)
group_a = np.random.normal(50, 10, n)

# Group B: shifted by +5 with correlated noise
group_b = group_a + np.random.normal(5, 8, n)

print(f"Sample size: {n}")
print(f"Group A: mean={group_a.mean():.2f}, SD={group_a.std():.2f}")
print(f"Group B: mean={group_b.mean():.2f}, SD={group_b.std():.2f}")
print(f"Mean difference (B - A): {(group_b - group_a).mean():.2f}")

### 2.2 Generate all four plots with `plot_array()`

In [ ]:
# Generate all four EZ-Pair Graph visualizations
figures = ezpg.plot_array(
    group_a, group_b,
    output_dir="output_demo",
    format="png"
)

# Display generated plots inline
from IPython.display import Image as IPImage, display
for name, path in figures.items():
    print(f"\n{name}:")
    display(IPImage(filename=path, width=600))


### 2.3 Display individual plots inline

You can also use the lower-level plotting functions for more control.

In [ ]:
from ez_pair_graph.plotting import plot_slopegraph, plot_trapezoid, plot_clustered_lines, plot_parallel_arrows
from ez_pair_graph.preparation import cluster_data, compute_statistics

# Step 1: Cluster the data
result = cluster_data(group_a, group_b, method="hierarchical", max_k=7)
clusters = result["clusters"]
print(f"Number of clusters: {result['n_clusters']}")

# Step 2: Compute statistics
stats = compute_statistics(group_a, group_b, clusters)

In [ ]:
# Slope Graph
fig, ax = plot_slopegraph(group_a, group_b, alpha=0.2)
ax.set_title("Slope Graph — Synthetic Data (shift=+5)", fontsize=12)
plt.show()

In [ ]:
# Trapezoid Plot
fig, ax = plot_trapezoid(group_a, group_b)
ax.set_title("Trapezoid Plot — Synthetic Data (shift=+5)", fontsize=12)
plt.show()

In [ ]:
# Clustered Line Plot
fig, ax = plot_clustered_lines(group_a, group_b, clusters, stats)
ax.set_title("Clustered Line Plot — Synthetic Data (shift=+5)", fontsize=12)
plt.show()

In [ ]:
# Parallel Arrow Plot
fig, ax = plot_parallel_arrows(group_a, group_b, clusters, stats)
ax.set_title("Parallel Arrow Plot — Synthetic Data (shift=+5)", fontsize=12)
plt.show()

### 2.4 Comparison: No-shift condition

For contrast, here is the same visualization when there is no systematic difference between groups.

In [ ]:
# No-shift condition
np.random.seed(123)
group_a_null = np.random.normal(50, 10, n)
group_b_null = group_a_null + np.random.normal(0, 8, n)  # shift = 0

result_null = cluster_data(group_a_null, group_b_null)
clusters_null = result_null["clusters"]
stats_null = compute_statistics(group_a_null, group_b_null, clusters_null)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Trapezoid: shift vs no-shift
plot_trapezoid(group_a, group_b, ax=axes[0])
axes[0].set_title("Shift = +5")

plot_trapezoid(group_a_null, group_b_null, ax=axes[1])
axes[1].set_title("No Shift")

fig.suptitle("Trapezoid Plot Comparison", fontsize=14)
plt.tight_layout()
plt.show()

## 3. Working with DataFrames

`plot_dataframe()` accepts a pandas DataFrame directly.

In [ ]:
# Create a DataFrame
df = pd.DataFrame({
    "before_treatment": group_a,
    "after_treatment": group_b,
    "sample_id": [f"S{i+1}" for i in range(n)]
})

print(df.head(10))

In [ ]:
# Generate plots from DataFrame
figures = ezpg.plot_dataframe(
    df,
    x_col="before_treatment",
    y_col="after_treatment",
    output_dir="output_dataframe",
    format="png",
    plots=["trapezoid", "clustered_line"]
)

for name, path in figures.items():
    print(f"\n{name}:")
    display(IPImage(filename=path, width=600))


## 4. File-Based Workflow

The `plot()` function reads directly from CSV/TSV files, suitable for integration into analysis pipelines.

In [ ]:
# Save test data to CSV
df[["before_treatment", "after_treatment"]].to_csv("test_data.csv", index=False)

# Generate plots from file
figures = ezpg.plot(
    "test_data.csv",
    output_dir="output_from_file",
    format="png",
    plots=["slopegraph", "trapezoid"]
)

for name, path in figures.items():
    print(f"\n{name}:")
    display(IPImage(filename=path, width=600))


## 5. Clustering Methods

### 5.1 Hierarchical Clustering (default)

Uses Ward's method with automatic k selection via the elbow method.

### 5.2 HDBSCAN Clustering

Density-based clustering that automatically determines the number of clusters and can identify noise points.

In [ ]:
# Compare hierarchical vs HDBSCAN clustering
result_hier = cluster_data(group_a, group_b, method="hierarchical", max_k=7)
result_hdb = cluster_data(group_a, group_b, method="hdbscan", min_cluster_size=10)

print(f"Hierarchical: {result_hier['n_clusters']} clusters")
print(f"HDBSCAN:      {result_hdb['n_clusters']} clusters")

# Visualize both
stats_hier = compute_statistics(group_a, group_b, result_hier["clusters"])
stats_hdb = compute_statistics(group_a, group_b, result_hdb["clusters"])

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

plot_clustered_lines(group_a, group_b, result_hier["clusters"], stats_hier, ax=axes[0])
axes[0].set_title(f"Hierarchical ({result_hier['n_clusters']} clusters)")

plot_clustered_lines(group_a, group_b, result_hdb["clusters"], stats_hdb, ax=axes[1])
axes[1].set_title(f"HDBSCAN ({result_hdb['n_clusters']} clusters)")

fig.suptitle("Clustering Method Comparison", fontsize=14)
plt.tight_layout()
plt.show()

## 6. Additional Options

### Log2 transformation
Useful for gene expression data or other log-scale measurements.

### Outlier filtering
Removes data points outside the whisker range (Q1 − 1.5×IQR to Q3 + 1.5×IQR).

In [ ]:
# Example with log2 transformation (simulating expression data)
np.random.seed(99)
expr_a = np.random.lognormal(mean=5, sigma=1, size=200)
expr_b = expr_a * np.random.lognormal(mean=0.3, sigma=0.5, size=200)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

plot_slopegraph(expr_a, expr_b, ax=axes[0], alpha=0.15)
axes[0].set_title("Raw Scale")

plot_slopegraph(expr_a, expr_b, ax=axes[1], log2=True, alpha=0.15)
axes[1].set_title("Log2 Transformed")

fig.suptitle("Expression Data: Raw vs Log2", fontsize=14)
plt.tight_layout()
plt.show()

## 7. Large-Scale Data (n > 1,000)

EZ-Pair Graph is designed to handle large datasets. The trapezoid plot and clustered line plot provide clear summaries even with thousands of data points.

In [ ]:
# Large-scale example (n=5000)
np.random.seed(7)
n_large = 5000
x_large = np.random.normal(100, 20, n_large)
y_large = x_large + np.random.normal(3, 15, n_large)

figures = ezpg.plot_array(
    x_large, y_large,
    output_dir="output_large",
    format="png",
    show_numbers=True
)

for name, path in figures.items():
    print(f"\n{name}:")
    display(IPImage(filename=path, width=600))


## 8. Command-Line Interface

EZ-Pair Graph also provides a CLI for batch processing:

```bash
# Basic usage
ez-pair-graph data.txt

# PNG output with HDBSCAN clustering
ez-pair-graph data.txt --format png --method hdbscan

# Specific plots only
ez-pair-graph data.txt --plots trapezoid clustered_line

# With log2 transformation and outlier removal
ez-pair-graph data.txt --log2 --no-outliers

# Custom output directory
ez-pair-graph data.txt --output-dir my_results --output-prefix experiment1
```

## 9. Accessing Cluster Statistics

The `compute_statistics()` function returns detailed per-cluster statistics.

In [ ]:
# Examine cluster statistics
result = cluster_data(group_a, group_b)
stats = compute_statistics(group_a, group_b, result["clusters"])

for cluster_id in sorted(stats.keys()):
    s = stats[cluster_id]
    n = s["calculated"]["n"]
    diff_mean = s["diff"]["mean"]
    x_med = s["x"]["q2"]
    y_med = s["y"]["q2"]
    print(f"Cluster {cluster_id}: n={n:3d}, X median={x_med:.1f}, Y median={y_med:.1f}, "
          f"mean diff={diff_mean:+.2f}")

## Summary

| Function | Input | Description |
|----------|-------|-------------|
| `ezpg.plot(file)` | File path | Full pipeline from file |
| `ezpg.plot_array(x, y)` | NumPy arrays | Full pipeline from arrays |
| `ezpg.plot_dataframe(df)` | DataFrame | Full pipeline from DataFrame |
| `ezpg.cluster_data(x, y)` | Arrays | Clustering only |
| `ezpg.compute_statistics(x, y, clusters)` | Arrays + clusters | Statistics only |

For more details, see the [GitHub repository](https://github.com/010049nn/EZ_pair_graph) and the accompanying paper.

---
*Copyright (c) 2025. RIKEN All rights reserved. Academic and non-commercial research use only.*